In [ ]:
# pip install "kserve<0.18"
#!pip install -U \
#    kfp==2.16.1 \
#    model-registry==0.3.9 \


In [1]:
import os
import kfp

TOKEN_PATH = os.environ.get(
    "KF_PIPELINES_SA_TOKEN_PATH",
    "/var/run/secrets/ml-pipeline/token",
)

KFP_HOST = os.environ.get(
    "KF_PIPELINES_ENDPOINT",
    "http://ml-pipeline.kubeflow.svc.cluster.local:8888",
)

token = open(TOKEN_PATH).read().strip()

print(f"Token present: {len(token)} chars")

client = kfp.Client(
    host=KFP_HOST,
    existing_token=token,
)

print("KFP SDK:", kfp.__version__)
print("Healthz:", client.get_kfp_healthz())

Token present: 1154 chars
KFP SDK: 2.16.1
Healthz: {'multi_user': True, 'pipeline_store': 'database'}


/opt/conda/lib/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


In [2]:
from kfp import dsl
from kfp.dsl import (
    component,
    Input,
    Output,
    Dataset,
    Model,
    Metrics,
)


@component(
    base_image="python:3.11-slim",
    packages_to_install=[
        "scikit-learn==1.4.2",
        "pandas==2.2.2",
    ],
)
def load_data(
    dataset: Output[Dataset],
):
    from sklearn.datasets import load_iris
    import pandas as pd

    iris = load_iris(as_frame=True)

    df = iris.frame

    df.to_csv(
        dataset.path,
        index=False,
    )

    print(f"Saved dataset to {dataset.path}")


@component(
    base_image="python:3.11-slim",
    packages_to_install=[
        "scikit-learn==1.4.2",
        "pandas==2.2.2",
        "joblib==1.4.2",
    ],
)
def train_model(
    dataset: Input[Dataset],
    model: Output[Model],
    metrics: Output[Metrics],
    n_neighbors: int = 5,
):
    import os
    import joblib
    import pandas as pd

    from sklearn.model_selection import train_test_split
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import accuracy_score

    df = pd.read_csv(dataset.path)

    X = df.drop(columns=["target"])
    y = df["target"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    clf = KNeighborsClassifier(
        n_neighbors=n_neighbors,
    )

    clf.fit(X_train, y_train)

    preds = clf.predict(X_test)

    acc = accuracy_score(
        y_test,
        preds,
    )

    metrics.log_metric(
        "accuracy",
        float(acc),
    )

    os.makedirs(
        model.path,
        exist_ok=True,
    )

    model_file = os.path.join(
        model.path,
        "model.joblib",
    )

    joblib.dump(
        clf,
        model_file,
    )

    print("Model path:", model.path)
    print("Model URI:", model.uri)
    print("Saved:", model_file)


@component(
    base_image="python:3.11-slim",
    packages_to_install=[
        "scikit-learn==1.4.2",
        "pandas==2.2.2",
        "joblib==1.4.2",
    ],
)
def evaluate_model(
    model: Input[Model],
    dataset: Input[Dataset],
    metrics: Output[Metrics],
):
    import os
    import joblib
    import pandas as pd

    from sklearn.metrics import (
        classification_report,
        accuracy_score,
    )

    df = pd.read_csv(dataset.path)

    X = df.drop(columns=["target"])
    y = df["target"]

    clf = joblib.load(
        os.path.join(
            model.path,
            "model.joblib",
        )
    )

    preds = clf.predict(X)

    acc = accuracy_score(
        y,
        preds,
    )

    print(
        classification_report(
            y,
            preds,
        )
    )

    metrics.log_metric(
        "full_dataset_accuracy",
        float(acc),
    )


@component(
    base_image="python:3.11-slim",
    packages_to_install=[
        "model-registry==0.3.7",
    ],
)
def register_model(
    model: Input[Model],
):
    from model_registry import ModelRegistry

    registry = ModelRegistry(
        server_address="http://model-registry-service",
        port=8080,
        author="kubeflow-demo",
        is_secure=False,
    )

    print("Registering:", model.uri)

    registered = registry.register_model(
        name="iris-knn",
        uri=model.uri,
        version="v1.0.0",
        description="KNN classifier on Iris dataset",
        model_format_name="sklearn",
        model_format_version="1",
        metadata={
            "framework": "scikit-learn",
            "dataset": "iris",
        },
    )

    print("Registered model:", registered)

In [3]:
@dsl.pipeline(
    name="iris-knn-pipeline",
    description="Load -> Train -> Evaluate -> Register",
)
def iris_pipeline(
    n_neighbors: int = 5,
):

    load_task = load_data()

    train_task = train_model(
        dataset=load_task.outputs["dataset"],
        n_neighbors=n_neighbors,
    )

    evaluate_task = evaluate_model(
        model=train_task.outputs["model"],
        dataset=load_task.outputs["dataset"],
    )

    register_task = register_model(
        model=train_task.outputs["model"],
    )

    register_task.after(
        evaluate_task,
    )

In [4]:
from kfp import compiler

PIPELINE_FILE = "/tmp/iris_pipeline.yaml"

compiler.Compiler().compile(
    pipeline_func=iris_pipeline,
    package_path=PIPELINE_FILE,
)

print(f"Pipeline compiled to {PIPELINE_FILE}")

run = client.create_run_from_pipeline_package(
    pipeline_file=PIPELINE_FILE,
    arguments={
        "n_neighbors": 5,
    },
    run_name="iris-knn-run-01",
    enable_caching=False,
)

print(f"Run submitted: {run.run_id}")

Pipeline compiled to /tmp/iris_pipeline.yaml


Run submitted: 98843421-dd88-41d9-9fb8-e44d7d93a03b


In [5]:
import time

run_id = run.run_id

while True:

    status = client.get_run(run_id).state

    print("Status:", status)

    if status in (
        "SUCCEEDED",
        "FAILED",
        "ERROR",
        "SKIPPED",
    ):
        break

    time.sleep(15)

assert status == "SUCCEEDED"

print("Pipeline SUCCEEDED")

Status: PENDING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: SUCCEEDED
Pipeline SUCCEEDED


In [7]:
from model_registry import ModelRegistry

registry = ModelRegistry(
    server_address="http://model-registry-service",
    port=8080,
    author="kubeflow-demo",
    is_secure=False,
)

version = registry.get_model_version(
    "iris-knn",
    "v1.0.0",
)

artifact = registry.get_model_artifact(
    "iris-knn",
    "v1.0.0",
)

print("Version:", version)
print("Artifact:", artifact)

Version: name='v1.0.0' id='2' description='KNN classifier on Iris dataset' external_id=None create_time_since_epoch='1780173963369' last_update_time_since_epoch='1780173963369' custom_properties={'dataset': 'iris', 'framework': 'scikit-learn'} author='kubeflow-demo' state=<ModelVersionState.LIVE: 'LIVE'> registered_model_id='1'
Artifact: name='iris-knn' id='1' description=None external_id=None create_time_since_epoch='1780173963384' last_update_time_since_epoch='1780173963384' custom_properties=None state=<ArtifactState.UNKNOWN: 'UNKNOWN'> experiment_id=None experiment_run_id=None model_format_name='sklearn' model_format_version='1' storage_key=None storage_path=None service_account_name=None model_source_kind=None model_source_class=None model_source_group=None model_source_id=None model_source_name=None uri='minio://mlpipeline/private-artifacts/kubeflow-user-example-com/v2/artifacts/iris-knn-pipeline/98843421-dd88-41d9-9fb8-e44d7d93a03b/train-model/dbd64371-7f9d-4ab7-985a-4d1e85d8d19

In [6]:
import kserve
from kubernetes import client as k8s_client
from model_registry import ModelRegistry

NAMESPACE = "kubeflow-user-example-com"
MODEL_NAME = "iris-knn"
MODEL_VERSION = "v1.0.0"
ISVC_NAME = "iris-knn-sklearn"

# ---------------------------------------------------
# Connect to Model Registry
# ---------------------------------------------------

registry = ModelRegistry(
    server_address="http://model-registry-service",
    port=8080,
    author="kubeflow-demo",
    is_secure=False,
)

model = registry.get_registered_model(MODEL_NAME)

version = registry.get_model_version(
    MODEL_NAME,
    MODEL_VERSION,
)

artifact = registry.get_model_artifact(
    MODEL_NAME,
    MODEL_VERSION,
)

print("Model ID:", model.id)
print("Version ID:", version.id)
print("Artifact URI:", artifact.uri)

# ---------------------------------------------------
# Convert registry URI to KServe-compatible S3 URI
# ---------------------------------------------------

storage_uri = artifact.uri.replace(
    "minio://",
    "s3://",
)

print("KServe Storage URI:", storage_uri)

# ---------------------------------------------------
# Delete old ISVC if it exists
# ---------------------------------------------------

client = kserve.KServeClient()

try:
    client.delete(
        name=ISVC_NAME,
        namespace=NAMESPACE,
    )
    print(f"Deleted existing InferenceService '{ISVC_NAME}'")
except Exception:
    pass

# ---------------------------------------------------
# Create InferenceService
# ---------------------------------------------------

isvc = kserve.V1beta1InferenceService(
    api_version="serving.kserve.io/v1beta1",
    kind="InferenceService",
    metadata=k8s_client.V1ObjectMeta(
        name=ISVC_NAME,
        namespace=NAMESPACE,
        labels={
            "modelregistry/registered-model-id": str(model.id),
            "modelregistry/model-version-id": str(version.id),
        },
    ),
    spec=kserve.V1beta1InferenceServiceSpec(
        predictor=kserve.V1beta1PredictorSpec(
            service_account_name="default-editor",
            model=kserve.V1beta1ModelSpec(
                model_format=kserve.V1beta1ModelFormat(
                    name=artifact.model_format_name,
                    version=artifact.model_format_version,
                ),
                runtime="kserve-sklearnserver",
                storage_uri=storage_uri,
            ),
        )
    ),
)

client.create(isvc)

print(f"InferenceService '{ISVC_NAME}' created")

Model ID: 1
Version ID: 2
Artifact URI: minio://mlpipeline/private-artifacts/kubeflow-user-example-com/v2/artifacts/iris-knn-pipeline/98843421-dd88-41d9-9fb8-e44d7d93a03b/train-model/dbd64371-7f9d-4ab7-985a-4d1e85d8d19e/model
KServe Storage URI: s3://mlpipeline/private-artifacts/kubeflow-user-example-com/v2/artifacts/iris-knn-pipeline/98843421-dd88-41d9-9fb8-e44d7d93a03b/train-model/dbd64371-7f9d-4ab7-985a-4d1e85d8d19e/model
InferenceService 'iris-knn-sklearn' created


In [ ]:
import json
import requests

NAMESPACE = "kubeflow-user-example-com"
ISVC_NAME = "iris-knn-sklearn"
MODEL_NAME = ISVC_NAME

# In-cluster inference via the Knative cluster-local-gateway.
# No bearer token is needed — the gateway has an allow-all AuthorizationPolicy
# and the predictor AP allows the Knative activator SA.
#
# One-time setup per user namespace (add to your profile/PodDefault setup):
#   kubectl apply -f allow-knative-serving.yaml
#
# allow-knative-serving.yaml:
#   apiVersion: security.istio.io/v1beta1
#   kind: AuthorizationPolicy
#   metadata:
#     name: allow-knative-serving
#     namespace: <user-namespace>
#   spec:
#     action: ALLOW
#     rules:
#     - from:
#       - source:
#           principals:
#           - cluster.local/ns/knative-serving/sa/activator
#           - cluster.local/ns/knative-serving/sa/controller

url = (
    f"http://{ISVC_NAME}.{NAMESPACE}.svc.cluster.local"
    f"/v1/models/{MODEL_NAME}:predict"
)

payload = {
    "instances": [
        [6.8, 2.8, 4.8, 1.4],
        [6.0, 3.4, 4.5, 1.6],
    ]
}

resp = requests.post(
    url,
    headers={"Content-Type": "application/json"},
    json=payload,
    timeout=60,
)

print(f"HTTP {resp.status_code}")
resp.raise_for_status()
print("Predictions:", resp.json())


In [ ]:
import json, requests

resp = requests.post(
    "http://iris-knn-sklearn.kubeflow-user-example-com.svc.cluster.local"
    "/v1/models/iris-knn-sklearn:predict",
    headers={"Content-Type": "application/json"},
    json={"instances": [[6.8, 2.8, 4.8, 1.4]]},
)
print(resp.json())